In [1]:
import os
import requests
import psycopg2
from datetime import date
from dotenv import load_dotenv
import subprocess

# =========================
# Setup inicial
# =========================

load_dotenv()

caffeinate_process = subprocess.Popen(['caffeinate'])

dbname = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
API_KEY = os.getenv("FMP_API_KEY")

conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)
cursor = conn.cursor()

# =========================
# Obtener tickers de universos
# =========================

def obtener_tickers_universos():
    sql = """
        SELECT DISTINCT ticker
        FROM universos.all_exchanges;
    """
    cursor.execute(sql)
    return {row[0] for row in cursor.fetchall()}

# =========================
# Obtener stock list FMP
# =========================

def obtener_stock_list():
    url = "https://financialmodelingprep.com/api/v3/stock/list"
    params = {"apikey": API_KEY}

    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    return response.json()

# =========================
# Filtrar solo tickers USA
# =========================

def filtrar_all_usa_list(stock_list, tickers_universo):
    registros = []

    for item in stock_list:
        symbol = item.get("symbol")

        if symbol and symbol in tickers_universo:
            registros.append(
                (
                    symbol,
                    item.get("name"),
                    item.get("exchangeShortName"),
                    item.get("type")
                )
            )

    return registros

# =========================
# Insertar en DB
# =========================

def insertar_all_usa_list(registros):
    hoy = date.today()

    values = [
        (ticker, name_stock, exchange_short, type_asset, hoy, "FMP")
        for ticker, name_stock, exchange_short, type_asset in registros
    ]

    sql = """
        INSERT INTO api_raw.all_usa_list (
            ticker,
            name_stock,
            exchange_short_name,
            type,
            fecha,
            source
        )
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (ticker, fecha) DO NOTHING;
    """

    cursor.executemany(sql, values)
    conn.commit()

# =========================
# Proceso principal
# =========================

def ejecutar_ingesta():
    print("📥 Obteniendo tickers de universos.all_exchanges...")
    tickers_universo = obtener_tickers_universos()
    print(f"✅ Tickers universo: {len(tickers_universo)}")

    print("📥 Descargando stock list completa de FMP...")
    stock_list = obtener_stock_list()
    print(f"✅ Registros stock/list: {len(stock_list)}")

    print("🔍 Filtrando solo tickers USA...")
    registros = filtrar_all_usa_list(stock_list, tickers_universo)
    print(f"✅ Registros finales all_usa_list: {len(registros)}")

    if not registros:
        print("⚠️ No hay datos para insertar.")
        return

    insertar_all_usa_list(registros)
    print("💾 api_raw.all_usa_list actualizada correctamente.")

# =========================
# Lanzar
# =========================

if __name__ == "__main__":
    try:
        ejecutar_ingesta()
    finally:
        caffeinate_process.terminate()
        cursor.close()
        conn.close()


📥 Obteniendo tickers de universos.all_exchanges...
✅ Tickers universo: 21880
📥 Descargando stock list completa de FMP...
✅ Registros stock/list: 87531
🔍 Filtrando solo tickers USA...
✅ Registros finales all_usa_list: 21880
💾 api_raw.all_usa_list actualizada correctamente.
